In [ ]:
import numpy as np
import pandas as pd
import configparser
import os  # Import os library for path operations

# Read configuration from config.ini
config = configparser.ConfigParser()
config.read('config.ini')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf

from utilsforecast.losses import mse
from utilsforecast.evaluation import evaluate

In [ ]:
def prepare_data(file_path):
    """
    Read CSV file and perform necessary preprocessing.

    Parameters:
    - file_path (str): Path to the CSV file.

    Returns:
    - pd.DataFrame: Processed DataFrame.
    """

    # Ensure the file path exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")

    # Define data types and index column
    data_types = {
        'ds': 'str',
        'unique_id': 'category',
        'y': np.int64
    }
    index_col = 'ds'

    # Read CSV with specified data types and index
    df = pd.read_csv(file_path, dtype=data_types, parse_dates=[index_col])

    # Check if 'ds' is present in the columns
    if 'ds' not in df.columns:
        raise KeyError("The 'ds' column is not present in the DataFrame.")

    # Convert 'ds' column to datetime after reading
    df['ds'] = pd.to_datetime(df['ds'], format='%Y-%m')  # Adjust the format based on your actual data
    
    df = df[['unique_id', 'ds', 'y']]
    
    return df

In [ ]:
# Example usage:
df_path = config['Paths']['dataset_data']
df = prepare_data(df_path)


In [ ]:
df.head(), df.shape

## Forecasting with AutoARIMA and AutoETS

### Start the models

In [ ]:
from statsforecast.models import (
    AutoARIMA,
    AutoETS
)
from statsforecast import StatsForecast

In [ ]:
models= [
    AutoARIMA(season_length=12),
    AutoETS(season_length=12)
]

In [ ]:
# Instantiate StatsForecast class as sf
sf = StatsForecast(
    df=df, 
    models=models,
    freq='MS', 
    n_jobs=-1
)

In [ ]:
forecast_df = sf.forecast(h=12, level=[90])
forecast_df.head()

In [ ]:
sf.plot(df, forecast_df, engine='matplotlib')

In [ ]:
forecast_df.to_csv(r"C:\Users\brvn\Documents\github\projects\HC\tcc_brvn\data\interim\forecasts_df.csv", float_format="%d")

In [ ]:
crossvaldation_df = sf.cross_validation(
    df=df,
    h=12,
    step_size=12,
    n_windows=1
  )

In [ ]:
crossvaldation_df.head()
# crossvaldation_df.to_csv(r"C:\Users\brvn\Documents\github\projects\HC\tcc_brvn\data\processed\cross-validation-df.csv")


In [ ]:
def evaluate_cross_validation(df, metric):
    models = df.drop(columns=['unique_id', 'ds', 'cutoff', 'y']).columns.tolist()
    evals = []
    # Calculate loss for every unique_id and cutoff.    
    for cutoff in df['cutoff'].unique():
        eval_ = evaluate(df[df['cutoff'] == cutoff], metrics=[metric], models=models)
        evals.append(eval_)
    evals = pd.concat(evals)
    evals = evals.groupby('unique_id', observed=False).mean(numeric_only=True) # Averages the error metrics for all cutoffs for every combination of model and unique_id
    evals['best_model'] = evals.idxmin(axis=1)
    return evals


In [ ]:
evaluation_df = evaluate_cross_validation(crossvaldation_df.reset_index(), mse)

evaluation_df.head()

In [ ]:
# Salvar o dataframe com o melhor modelo
evaluation_df.to_csv(r"C:\Users\brvn\Documents\github\projects\HC\tcc_brvn\data\processed\best-models-df.csv")

In [ ]:
#Criando um sumário dos melhores modelos
summary_df = evaluation_df.groupby('best_model').size().sort_values().to_frame()

summary_df.reset_index().columns = ["Model", "Nr. of unique_ids"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming summary_df is the DataFrame
model_counts = evaluation_df['best_model'].value_counts()
model_counts.plot(kind='bar')

# Customize the plot
plt.title('Best Models for Time Series Forecasting')
plt.xlabel('Model')
plt.ylabel('Count')
plt.xticks(rotation=0)  # Rotate x-axis labels for better readability

plt.show()


In [ ]:
autoets_ids = evaluation_df.query('best_model == "AutoETS"').index
autoarima_ids = evaluation_df.query('best_model == "AutoARIMA"').index


In [ ]:
sf.plot(df=df,forecasts_df=forecast_df, unique_ids=autoarima_ids, engine='matplotlib')

In [ ]:
sf.plot(df=df,
        forecasts_df=forecast_df, 
        unique_ids=autoets_ids, 
        engine='matplotlib')

In [ ]:
print(autoets_ids)

In [ ]:
print(autoarima_ids)

In [ ]:
def get_best_model_forecast(forecasts_df, evaluation_df):
    df = forecasts_df.set_index('ds', append=True).stack().to_frame().reset_index(level=2) # Wide to long 
    df.columns = ['model', 'best_model_forecast'] 
    df = df.join(evaluation_df[['best_model']])
    df = df.query('model.str.replace("-lo-90|-hi-90", "", regex=True) == best_model').copy()
    df.loc[:, 'model'] = [model.replace(bm, 'best_model') for model, bm in zip(df['model'], df['best_model'])]
    df = df.drop(columns='best_model').set_index('model', append=True).unstack()
    df.columns = df.columns.droplevel()
    df = df.reset_index(level=1)
    return df


In [ ]:
prod_forecasts_df = get_best_model_forecast(forecast_df, evaluation_df)

# prod_forecasts_df.to_csv(r"C:\Users\brvn\Documents\github\projects\HC\tcc_brvn\data\processed\forecasted-df.csv", float_format="%d")


In [ ]:
sf.plot(df, prod_forecasts_df, level=[90], engine='matplotlib')
